In [9]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("spam_train_cleaned.csv")
test_df = pd.read_csv("spam_test_cleaned.csv")

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

Train shape: (31311, 16)
Test shape : (7828, 16)

Train columns:
['label', 'urls', 'hour', 'combined_text', 'capital_letter_count', 'capital_ratio', 'exclamation_count', 'question_count', 'special_char_count', 'day_of_week_Friday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']


,label,urls,hour,combined_text,capital_letter_count,capital_ratio,exclamation_count,question_count,special_char_count,day_of_week_Friday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday
0,0,1,2,volunteers are needed for alumni phonothon 200...,37,0.055306,0,0,2,0,0,0,0,0,0,1
1,0,0,0,re opensuse opensuse and faxes on sunday 10 fe...,85,0.048935,0,2,42,0,0,0,0,0,0,1
2,0,1,2,re r matching a period in grep on 08 05 2008 0...,69,0.045128,0,4,114,0,0,0,0,0,0,1
3,1,1,17,fast and safe male enhancement huge love gun i...,11,0.034700,3,0,0,0,0,0,0,1,0,0
4,0,1,3,re python dev documentation reorganization was...,44,0.026113,0,0,94,0,0,0,0,0,0,1


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix


# X and y


X = train_df.drop(columns=["label"])
y = train_df["label"]

X_test_raw = test_df.drop(columns=["label"])
y_test = test_df["label"]



# Train / Validation split


X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



# Text + numerical columns


TEXT_COL = "combined_text"

NUMERIC_COLS = [
    col for col in X.columns
    if col != TEXT_COL
]



# TF-IDF


tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(X_train_raw[TEXT_COL])
X_text_val = tfidf.transform(X_val_raw[TEXT_COL])
X_text_test = tfidf.transform(X_test_raw[TEXT_COL])



# Scaling


scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_train_raw[NUMERIC_COLS])
X_num_val = scaler.transform(X_val_raw[NUMERIC_COLS])
X_num_test = scaler.transform(X_test_raw[NUMERIC_COLS])



# Combine ALL features


X_train = hstack([
    X_text_train,
    csr_matrix(X_num_train)
])

X_val = hstack([
    X_text_val,
    csr_matrix(X_num_val)
])

X_test = hstack([
    X_text_test,
    csr_matrix(X_num_test)
])


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (25048, 314)
Validation: (6263, 314)
Test: (7828, 314)


In [11]:
#Default SVM no hyperpramters
from sklearn.svm import SVC

svm_default = SVC()

svm_default.fit(X_train, y_train)

print("Default SVM trained!")

Default SVM trained!


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_train_pred = svm_default.predict(X_train)
y_val_pred = svm_default.predict(X_val)

print("========== DEFAULT SVM ==========")

print("\nTRAINING PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_train, y_train_pred), 4))
print("Precision:", round(precision_score(y_train, y_train_pred), 4))
print("Recall   :", round(recall_score(y_train, y_train_pred), 4))
print("F1 Score :", round(f1_score(y_train, y_train_pred), 4))

print("\nVALIDATION PERFORMANCE")
print("Accuracy :", round(accuracy_score(y_val, y_val_pred), 4))
print("Precision:", round(precision_score(y_val, y_val_pred), 4))
print("Recall   :", round(recall_score(y_val, y_val_pred), 4))
print("F1 Score :", round(f1_score(y_val, y_val_pred), 4))

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_val, y_val_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_val, y_val_pred))

========== DEFAULT SVM ==========

TRAINING PERFORMANCE
Accuracy : 0.986
Precision: 0.9835
Recall   : 0.9916
F1 Score : 0.9875

VALIDATION PERFORMANCE
Accuracy : 0.9823
Precision: 0.9801
Recall   : 0.9883
F1 Score : 0.9842

CONFUSION MATRIX
[[2700   70]
 [  41 3452]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      2770
           1       0.98      0.99      0.98      3493

    accuracy                           0.98      6263
   macro avg       0.98      0.98      0.98      6263
weighted avg       0.98      0.98      0.98      6263



In [13]:
#Hyperpramaters C
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

C_values = [0.01, 0.1, 1, 10, 100]

C_results = []

for C in C_values:

    model = SVC(
        C=C,
        kernel="rbf",
        gamma="scale"
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    C_results.append({
        "C": C,
        "Train Accuracy": accuracy_score(y_train, train_pred),
        "Validation Accuracy": accuracy_score(y_val, val_pred),
        "Train F1": f1_score(y_train, train_pred),
        "Validation F1": f1_score(y_val, val_pred),
        "F1 Gap": (
            f1_score(y_train, train_pred)
            - f1_score(y_val, val_pred)
        )
    })

C_results_df = pd.DataFrame(C_results)

display(C_results_df)



,C,Train Accuracy,Validation Accuracy,Train F1,Validation F1,F1 Gap
0,0.01,0.866576,0.873543,0.892291,0.897489,-0.005198
1,0.10,0.967263,0.968545,0.971033,0.972092,-0.001059
2,1.00,0.985987,0.982277,0.987487,0.984177,0.003310
3,10.00,0.996127,0.988185,0.996531,0.989432,0.007099
4,100.00,0.999800,0.988185,0.999821,0.989432,0.010389


So C=10 gives you the best validation result without the unnecessary complexity of C=100.

In [14]:
from sklearn.svm import SVC

svm_tuned = SVC(
    C=10,
    kernel="rbf",
    gamma="scale"
)

svm_tuned.fit(X_train, y_train)

y_train_pred_tuned = svm_tuned.predict(X_train)
y_val_pred_tuned = svm_tuned.predict(X_val)

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

comparison = pd.DataFrame({
    "Model": ["Default SVM (C=1)", "Tuned SVM (C=10)"],

    "Train Accuracy": [
        accuracy_score(y_train, y_train_pred),
        accuracy_score(y_train, y_train_pred_tuned)
    ],

    "Validation Accuracy": [
        accuracy_score(y_val, y_val_pred),
        accuracy_score(y_val, y_val_pred_tuned)
    ],

    "Train Precision": [
        precision_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred_tuned)
    ],

    "Validation Precision": [
        precision_score(y_val, y_val_pred),
        precision_score(y_val, y_val_pred_tuned)
    ],

    "Train Recall": [
        recall_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred_tuned)
    ],

    "Validation Recall": [
        recall_score(y_val, y_val_pred),
        recall_score(y_val, y_val_pred_tuned)
    ],

    "Train F1": [
        f1_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred_tuned)
    ],

    "Validation F1": [
        f1_score(y_val, y_val_pred),
        f1_score(y_val, y_val_pred_tuned)
    ]
})

display(comparison.round(4))

,Model,Train Accuracy,Validation Accuracy,Train Precision,Validation Precision,Train Recall,Validation Recall,Train F1,Validation F1
0,Default SVM (C=1),0.9860,0.9823,0.9835,0.9801,0.9916,0.9883,0.9875,0.9842
1,Tuned SVM (C=10),0.9961,0.9882,0.9957,0.9872,0.9974,0.9917,0.9965,0.9894


he default SVM generalized well because its training F1 (98.75%) and validation F1 (98.42%) were very close, with a difference of only 0.33 percentage points. After tuning C from 1 to 10, training F1 increased to 99.65% and validation F1 increased to 98.94%. Although the gap increased to 0.71 percentage points, validation performance also improved, indicating that the model was not simply overfitting. When C was increased further to 100, training F1 reached 99.98% while validation F1 remained at 98.94%, showing that additional complexity no longer improved generalization. Therefore, C=10 was selected.